In [0]:
# ================== Fourth take home assignment GR5069 Spring 2026 ========

# ================== Global Definitions ====================================

# # Packages to import
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow.sklearn
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from pyspark.sql.functions import col, when
from pyspark.sql.types import DoubleType
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import os 
import tempfile


# Data Paths 
sprint_results_path = "/Volumes/gr5069/raw/f1_data/sprint_results.csv"



In [0]:
# Read in the sprint dataset as a spark dataframe
sprint_df = spark.read.csv(sprint_results_path, header = True)

# Display data
sprint_df.count()
display(sprint_df)


In [0]:
# Format columns to use as training/testing dataset

# Replace "\N" with NA before converting to numeric 
sprint_df = sprint_df.withColumn("milliseconds", when(col("milliseconds") == r"\N", None).otherwise(col("milliseconds")))

# Convert milliseconds and laps column to numeric 
sprint_df = sprint_df.withColumn("milliseconds", col("milliseconds").cast(DoubleType()))
sprint_df = sprint_df.withColumn("laps", col("laps").cast(DoubleType()))

# Filter for only rows with non-null values in milliseconds and laps
sprint_df = sprint_df.filter(col("laps").isNotNull() & col("milliseconds").isNotNull())

In [0]:
# Display spark dataframe 
display(sprint_df)

In [0]:
# Display summary stats
display(sprint_df.select("milliseconds", "laps").describe())

#### 2. Build two (2) predictive models using MLflow, logging hyperparameters, the model itself, four metrics, and two artifacts. Submit your MLflow experiments as part of your assignments.

**Model 1: Random Forest Regression**

--------------------------------


This random forest regression model predicts the number of laps a driver completes based on their race time in milliseconds. The four hyperparameters for tuning the model are the following: 

-   n_estimators
-   max_depth
-   random_state
-  max_leaf_nodes 


The model which performed the best was model 5. with the following metrics listed: 

<br>

```
  mse: 1.4836542048363839
  mae: 0.7017585123798887
  r2: 0.8766244275995201
  rmse: 1.218053449088497
```

<br>

Model 5 received the lowest mean squared error of 1.48. This model received an R2 score of 0.876, indicating that this model explains 87% of variation. I've selected this model because it scored the lower MSE metric out of all five random forest regression models. 


In [0]:
# Train/Test/Split 

(trainDF, testDF) = sprint_df.randomSplit([.8, .2], seed=42)
print(trainDF.count())


In [0]:
# Define Basic RF regression model (with no defined parameters)

with mlflow.start_run(run_name="Baseline Model") as run:

    # Convert Spark DataFrames to Pandas
    trainPDF = trainDF.select("milliseconds", "laps").toPandas()
    testPDF = testDF.select("milliseconds", "laps").toPandas()

    X_train = trainPDF[["milliseconds"]]
    y_train = trainPDF["laps"]
    X_test = testPDF[["milliseconds"]]
    y_test = testPDF["laps"]

    # Create model, train it, and create predictions
    rf = RandomForestRegressor()
    rf.fit(X_train, y_train)
    predictions = rf.predict(X_test)

    # Log model
    mlflow.sklearn.log_model(rf, "random-forest-regressor-model")

    # Create metrics
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    rmse = np.sqrt(mse)

    # Log metrics
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("rmse", rmse)

    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

In [0]:
# Define function that logs parameters, metrics, and feature importance
def rf_predict(experimentID, run_name, params, trainDF, testDF, feature_col, target_col):
    import os
    import matplotlib.pyplot as plt
    import mlflow.sklearn
    import seaborn as sns
    import pandas as pd
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    import numpy as np
    import tempfile

    # Convert Spark DataFrames to Pandas
    trainPDF = trainDF.select(feature_col, target_col).toPandas()
    testPDF = testDF.select(feature_col, target_col).toPandas()

    X_train = trainPDF[[feature_col]]
    y_train = trainPDF[target_col]
    X_test = testPDF[[feature_col]]
    y_test = testPDF[target_col]

    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
        # Create model, train it, and create predictions
        rf = RandomForestRegressor(**params)
        rf.fit(X_train, y_train)
        predictions = rf.predict(X_test)

        mlflow.sklearn.log_model(rf, "Basic random forest regressor model")

        # Log params
        [mlflow.log_param(param, value) for param, value in params.items()]

        # Create metrics
        mse = mean_squared_error(y_test, predictions)
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        rmse = np.sqrt(mse)
        print("  mse: {}".format(mse))
        print("  mae: {}".format(mae))
        print("  r2: {}".format(r2))
        print("  rmse: {}".format(rmse))

        # Log metrics
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("rmse", rmse)

        # Create feature importance table
        importance = pd.DataFrame(list(zip(X_train.columns, rf.feature_importances_)),
                                  columns=["Feature", "Importance"]
                                  ).sort_values("Importance", ascending=False)

        # Log feature importance using a temporary file
        temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
        temp_name = temp.name
        try:
            importance.to_csv(temp_name, index=False)
            mlflow.log_artifact(temp_name, "feature-importance.csv")
        finally:
            temp.close()

        # Create plot
        fig, ax = plt.subplots()

        sns.residplot(x=predictions, y=y_test.astype(float), lowess=True)
        plt.xlabel("Predicted Laps")
        plt.ylabel("Residual")
        plt.title("Residual Plot")

        # Log residuals using a temporary file
        temp = tempfile.NamedTemporaryFile(prefix="residuals-", suffix=".png")
        temp_name = temp.name
        try:
            fig.savefig(temp_name)
            mlflow.log_artifact(temp_name, "residuals.png")
        finally:
            temp.close()

        display(fig)
        plt.close(fig)
    return run.info.run_id

In [0]:
# First run
params = {
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": 42,
    "max_leaf_nodes": 10
}

rf_predict(experimentID, "1st Run", params, trainDF, testDF, "milliseconds", "laps")

In [0]:
# Second run 

params = {
    "n_estimators": 125,
    "max_depth": 7,
    "random_state": 42,
    "max_leaf_nodes": 10
}

rf_predict(experimentID, "2nd Run", params, trainDF, testDF, "milliseconds", "laps")

In [0]:
# Third run 

params = {
    "n_estimators": 75,
    "max_depth": 10,
    "random_state": 42,
    "max_leaf_nodes": 10
}

rf_predict(experimentID, "3rd Run", params, trainDF, testDF, "milliseconds", "laps")

In [0]:
# Fourth run 

params = {
    "n_estimators": 125,
    "max_depth": 7,
    "random_state": 42,
    "max_leaf_nodes": 10
}

rf_predict(experimentID, "4th Run", params, trainDF, testDF, "milliseconds", "laps")

In [0]:
# Fifth run 

params = {
    "n_estimators": 125,
    "max_depth": 10,
    "random_state": 42,
    "max_leaf_nodes": 15
}

rf_predict(experimentID, "5th Run", params, trainDF, testDF, "milliseconds", "laps")

In [0]:
# Write predictions table from best model as CSV to personal database 
runID = "243434143b33461bbecf07075a8b5194" # Run ID of best model (model 5)
model = mlflow.sklearn.load_model(f"runs:/{runID}/Basic random forest regressor model")

testPDF = testDF.toPandas()
testPDF["prediction"] = model.predict(testPDF[["milliseconds"]])

predDF = spark.createDataFrame(testPDF)
predDF.coalesce(1).write.mode('overwrite').csv('/Volumes/gr5069/sre2129/takehome/model1/model_1_prediction_table', header=True)

**Model 2. Gradient Boosted Regression**

------------------

This gradient boosted regression model predicts the number of laps a driver completes based on their race time in milliseconds. The four hyperparameters for tuning the model are the following: 

  - n_estimators
  - max_depth
  - random_state
  - learning_rate


The model which performed the best was model 5. with the following metrics listed: 

<br>

```
  mse: 1.6290999785428617
  mae: 0.6934939516975053
  r2: 0.8645296581271104
  rmse: 1.2763620092054062
```

<br>

Model 5 received the lowest mean squared error of 1.62. This model received an R2 score of 0.864, indicating that this model explains 86.4% of variation. I've selected this model because it scored the lowest MSE metric out of all five gradient boosted models. 


In [0]:
# Define Basic Gradient Boosting Regressor model (with no defined parameters)
with mlflow.start_run(run_name="Baseline Model - GBR") as run:

    # Convert Spark DataFrames to Pandas
    trainPDF = trainDF.select("milliseconds", "laps").toPandas()
    testPDF = testDF.select("milliseconds", "laps").toPandas()

    X_train = trainPDF[["milliseconds"]]
    y_train = trainPDF["laps"]
    X_test = testPDF[["milliseconds"]]
    y_test = testPDF["laps"]

    # Create model, train it, and create predictions
    gbr = GradientBoostingRegressor()
    gbr.fit(X_train, y_train)
    predictions = gbr.predict(X_test)

    # Log model
    mlflow.sklearn.log_model(gbr, "gradient-boosting-regressor-model")

    # Create metrics
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    rmse = np.sqrt(mse)

    # Log metrics
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("rmse", rmse)

    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

In [0]:

#Define function that logs parameters, metrics, and feature importance

def gbr_predict(experimentID, run_name, params, trainDF, testDF, feature_col, target_col):
    import os
    import matplotlib.pyplot as plt
    import mlflow.sklearn
    import seaborn as sns
    import pandas as pd
    from sklearn.ensemble import GradientBoostingRegressor
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    import numpy as np
    import tempfile

    # Convert Spark DataFrames to Pandas
    trainPDF = trainDF.select(feature_col, target_col).toPandas()
    testPDF = testDF.select(feature_col, target_col).toPandas()

    X_train = trainPDF[[feature_col]]
    y_train = trainPDF[target_col]
    X_test = testPDF[[feature_col]]
    y_test = testPDF[target_col]

    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
        # Create model, train it, and create predictions
        gbr = GradientBoostingRegressor(**params)
        gbr.fit(X_train, y_train)
        predictions = gbr.predict(X_test)

        mlflow.sklearn.log_model(gbr, "Basic gradient boosting regressor model")

        # Log params
        [mlflow.log_param(param, value) for param, value in params.items()]

        # Create metrics
        mse = mean_squared_error(y_test, predictions)
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        rmse = np.sqrt(mse)
        print("  mse: {}".format(mse))
        print("  mae: {}".format(mae))
        print("  r2: {}".format(r2))
        print("  rmse: {}".format(rmse))

        # Log metrics
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("rmse", rmse)

        # Create feature importance table
        importance = pd.DataFrame(list(zip(X_train.columns, gbr.feature_importances_)),
                                  columns=["Feature", "Importance"]
                                  ).sort_values("Importance", ascending=False)

        # Log feature importance using a temporary file
        temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
        temp_name = temp.name
        try:
            importance.to_csv(temp_name, index=False)
            mlflow.log_artifact(temp_name, "feature-importance.csv")
        finally:
            temp.close()

        # Create plot
        fig, ax = plt.subplots()

        sns.residplot(x=predictions, y=y_test.astype(float), lowess=True)
        plt.xlabel("Predicted Laps")
        plt.ylabel("Residual")
        plt.title("Residual Plot")

        # Log residuals using a temporary file
        temp = tempfile.NamedTemporaryFile(prefix="residuals-", suffix=".png")
        temp_name = temp.name
        try:
            fig.savefig(temp_name)
            mlflow.log_artifact(temp_name, "residuals.png")
        finally:
            temp.close()

        display(fig)
        plt.close(fig)

    return run.info.run_id

In [0]:
# First run 
params = {
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": 42,
    "learning_rate": 0.1
}

gbr_predict(experimentID, "1st Run", params, trainDF, testDF, "milliseconds", "laps")

In [0]:
# Second run 
params = {
    "n_estimators": 125,
    "max_depth": 5,
    "random_state": 42,
    "learning_rate": 1
}

gbr_predict(experimentID, "2nd Run", params, trainDF, testDF, "milliseconds", "laps")

In [0]:
# Third run 
params = {
    "n_estimators": 150,
    "max_depth": 10,
    "random_state": 42,
    "learning_rate": 0.01
}

gbr_predict(experimentID, "3rd Run", params, trainDF, testDF, "milliseconds", "laps")

In [0]:
# Fourth run 
params = {
    "n_estimators": 125,
    "max_depth": 3,
    "random_state": 42,
    "learning_rate": 0.001
}

gbr_predict(experimentID, "4th Run", params, trainDF, testDF, "milliseconds", "laps")

In [0]:
# Fifth run 
params = {
    "n_estimators": 100,
    "max_depth": 3,
    "random_state": 42,
    "learning_rate": 0.1
}

gbr_predict(experimentID, "5th Run", params, trainDF, testDF, "milliseconds", "laps")

In [0]:


runID = "934eb71670de48b7a6baf42f2dffcd65"
model = mlflow.sklearn.load_model(f"runs:/{runID}/Basic gradient boosting regressor model")

testPDF = testDF.toPandas()
testPDF["prediction"] = model.predict(testPDF[["milliseconds"]])

predDF = spark.createDataFrame(testPDF)
predDF.coalesce(1).write.mode('overwrite').csv('/Volumes/gr5069/sre2129/takehome/model2/model_2_prediction_table', header=True)